# Precipitable Water & MAC Calibration at Addis Ababa

**Core finding**: HIPS/FTIR ratio correlates with AERONET precipitable water (r≈−0.48, p<1e-10).  
This notebook develops the physical mechanism and tests whether PW can ground a site-specific seasonal MAC calibration.

## Structure
1. **The Money Plot** — Effective MAC vs Precipitable Water (per-sample)
2. **FTIR vs HIPS correlation gap with AOD** — formal statistical test
3. **Morning-weighting bias** — what fraction of 24h filter mass comes from 6–10am?
4. **PW as predictor of slope vs intercept** — does moisture affect MAC (slope) or baseline (intercept)?
5. **Synthesis** — toward a PW-adjusted MAC

**Narrative**: At Addis Ababa, HIPS-FTIR disagreement is driven by atmospheric moisture affecting optical absorption measurements. Column water vapor (from collocated AERONET) explains ~23% of the variance in HIPS/FTIR ratio. The site's extreme boundary layer dynamics at 2370m cause morning-dominated filter loading, and seasonal moisture variation drives MAC from ~12.8 (dry) to ~8.4 (Kiremt). A precipitable-water-adjusted MAC provides physically grounded site-specific calibration.

---

## Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

RESEARCH_DIR = '/Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem'
scripts_path = os.path.join(RESEARCH_DIR, 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, FILTER_DATA_PATH, PROCESSED_SITES_DIR, MAC_VALUE
from data_matching import (
    load_aethalometer_data, load_filter_data,
    match_all_parameters, pivot_filter_by_id
)

# Addis season definitions
SEASONS = {'Dry': [10, 11, 12, 1, 2], 'Belg': [3, 4, 5], 'Kiremt': [6, 7, 8, 9]}
SEASON_COLORS = {'Dry': '#E67E22', 'Belg': '#27AE60', 'Kiremt': '#3498DB'}
SEASON_ORDER = ['Dry', 'Belg', 'Kiremt']

def assign_season(month):
    for name, months in SEASONS.items():
        if month in months:
            return name
    return 'Unknown'

AERONET_MISSING = -999.
SITE_CODE = 'ETAD'

print(f'MAC_VALUE from config: {MAC_VALUE} m²/g')
print('Setup complete')

## Data Loading

In [ ]:
# === Paths ===
BC_MINUTE_PATH = (
    '/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/'
    'My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/'
    'Aethelometry Data/JacrosMA350 60s Data20250804082112/'
    'df_jacros_cleaned_API_and_OG_manual_BC_all_wl.pkl'
)

AERONET_BASE = (
    '/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/'
    'My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/'
    'AERONET/Jacros'
)
AERONET_AOD_DAILY = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET Daily',
    '20220101_20251231_AAU_Jackros_ET.lev20')
AERONET_SDA_DAILY = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET Daily',
    '20220101_20251231_AAU_Jackros_ET.ONEILL_lev20')
AERONET_AOD_ALLPTS = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET all',
    '20220101_20251231_AAU_Jackros_ET.lev20')

print('Paths configured')

In [ ]:
# === Load minute-level BC ===
bc_raw = pd.read_pickle(BC_MINUTE_PATH)
bc_raw['datetime_local'] = pd.to_datetime(bc_raw['datetime_local'])
bc_raw.set_index('datetime_local', inplace=True)
bc_raw.sort_index(inplace=True)

for col in ['UV BCc', 'IR BCc', 'UV BC1', 'IR BC1']:
    if col in bc_raw.columns:
        bc_raw[col] = bc_raw[col] / 1000  # ng/m³ → µg/m³

for col in ['UV BCc', 'IR BCc']:
    if col in bc_raw.columns:
        bc_raw.loc[bc_raw[col] < 0, col] = np.nan
        mu, sigma = bc_raw[col].mean(), bc_raw[col].std()
        bc_raw.loc[bc_raw[col] > mu + 3 * sigma, col] = np.nan

print(f'BC minute data: {len(bc_raw):,} records')
print(f'  Range: {bc_raw.index.min()} → {bc_raw.index.max()}')

In [ ]:
# === Load daily AERONET ===
def load_aeronet_daily(filepath, date_col='Date(dd:mm:yyyy)'):
    df = pd.read_csv(filepath, skiprows=6)
    df['Date'] = pd.to_datetime(df[date_col], format='%d:%m:%Y')
    df.set_index('Date', inplace=True)
    df.sort_index(inplace=True)
    df.replace(AERONET_MISSING, np.nan, inplace=True)
    return df

aod_daily = load_aeronet_daily(AERONET_AOD_DAILY)
sda_daily = load_aeronet_daily(AERONET_SDA_DAILY, date_col='Date_(dd:mm:yyyy)')

for df in [aod_daily, sda_daily]:
    df['month'] = df.index.month
    df['season'] = df['month'].map(assign_season)

# Also load all-points for sub-daily PW
aod_all = pd.read_csv(AERONET_AOD_ALLPTS, skiprows=6)
aod_all['datetime_utc'] = pd.to_datetime(
    aod_all['Date(dd:mm:yyyy)'] + ' ' + aod_all['Time(hh:mm:ss)'],
    format='%d:%m:%Y %H:%M:%S')
aod_all.set_index('datetime_utc', inplace=True)
aod_all.sort_index(inplace=True)
aod_all.replace(AERONET_MISSING, np.nan, inplace=True)

print(f'AERONET daily AOD: {len(aod_daily)} days')
print(f'AERONET daily SDA: {len(sda_daily)} days')
print(f'AERONET all-points AOD: {len(aod_all):,}')
print(f'PW range: {aod_daily["Precipitable_Water(cm)"].min():.2f} – {aod_daily["Precipitable_Water(cm)"].max():.2f} cm')

In [ ]:
# === Load 24h filter data ===
aeth_data = load_aethalometer_data()
filter_data = load_filter_data()

df_matched = match_all_parameters('Addis_Ababa', SITE_CODE,
                                   aeth_data['Addis_Ababa'], filter_data)

if df_matched is not None:
    df_matched['date'] = pd.to_datetime(df_matched['date'])
    df_matched['month'] = df_matched['date'].dt.month
    df_matched['season'] = df_matched['month'].map(assign_season)
    
    # Compute effective MAC per sample:
    # hips_fabs from match_all_parameters = HIPS_Fabs / MAC_VALUE (µg/m³ EC equivalent)
    # So actual HIPS_Fabs (absorption, 1/Mm) = hips_fabs * MAC_VALUE
    # Effective MAC = HIPS_Fabs / FTIR_EC = (hips_fabs * MAC_VALUE) / ftir_ec
    valid_both = df_matched.dropna(subset=['hips_fabs', 'ftir_ec'])
    valid_both = valid_both[valid_both['ftir_ec'] > 0]
    df_matched.loc[valid_both.index, 'effective_mac'] = (
        valid_both['hips_fabs'] * MAC_VALUE) / valid_both['ftir_ec']
    df_matched.loc[valid_both.index, 'hips_ftir_ratio'] = (
        valid_both['hips_fabs'] / valid_both['ftir_ec'])  # EC/EC ratio
    
    print(f'Matched filter samples: {len(df_matched)}')
    print(f'  With both HIPS & FTIR: {len(valid_both)}')
    print(f'  Effective MAC: mean={df_matched["effective_mac"].mean():.1f}, '
          f'median={df_matched["effective_mac"].median():.1f} m²/g')
    for col in ['ir_bcc', 'hips_fabs', 'ftir_ec', 'iron']:
        if col in df_matched.columns:
            n = df_matched[col].notna().sum()
            print(f'  {col}: {n} valid')
else:
    print('WARNING: No matched filter data')

In [ ]:
# === Merge filter data with daily AERONET ===
coarse_col = 'Coarse_Mode_AOD_500nm[tau_c]'
fine_col = 'Fine_Mode_AOD_500nm[tau_f]'
fmf_col = 'FineModeFraction_500nm[eta]'

# Combine daily AOD + SDA
aeronet_daily_combined = aod_daily[[
    'AOD_500nm', 'Precipitable_Water(cm)', '440-870_Angstrom_Exponent'
]].join(sda_daily[[coarse_col, fine_col, fmf_col]], how='outer')

# Merge with filter data by date
filt = df_matched.copy()
filt['date_key'] = filt['date'].dt.normalize()
aeronet_daily_combined.index = pd.to_datetime(aeronet_daily_combined.index)

filt = filt.merge(aeronet_daily_combined, left_on='date_key',
                  right_index=True, how='left')

n_pw = filt['Precipitable_Water(cm)'].notna().sum()
n_mac = filt['effective_mac'].notna().sum()
n_both = filt.dropna(subset=['effective_mac', 'Precipitable_Water(cm)']).shape[0]
print(f'Filter samples with PW: {n_pw} / {len(filt)}')
print(f'Filter samples with MAC: {n_mac} / {len(filt)}')
print(f'Filter samples with both: {n_both}')

---

## 1. The Money Plot: Effective MAC vs Precipitable Water

If atmospheric moisture alters the optical absorption properties measured by HIPS (filter-based light transmission) but not the thermal EC measured by FTIR, then the effective MAC (HIPS Fabs / FTIR EC) should track precipitable water.

A clean MAC–PW relationship provides a physical basis for site-specific seasonal MAC calibration: **a single MAC doesn't work at Addis because moisture varies seasonally.**

In [ ]:
# The Money Plot: Effective MAC vs Precipitable Water
valid_mac_pw = filt.dropna(subset=['effective_mac', 'Precipitable_Water(cm)']).copy()
# Remove extreme outliers for cleaner visualization
mac_q01 = valid_mac_pw['effective_mac'].quantile(0.01)
mac_q99 = valid_mac_pw['effective_mac'].quantile(0.99)
valid_mac_pw = valid_mac_pw[
    (valid_mac_pw['effective_mac'] >= mac_q01) &
    (valid_mac_pw['effective_mac'] <= mac_q99)]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Panel 1: Scatter with regression ---
ax = axes[0]
for season in SEASON_ORDER:
    s = valid_mac_pw[valid_mac_pw['season'] == season]
    ax.scatter(s['Precipitable_Water(cm)'], s['effective_mac'],
              color=SEASON_COLORS[season], alpha=0.6, s=60,
              label=season, edgecolors='k', linewidth=0.3)

r, p = stats.pearsonr(valid_mac_pw['Precipitable_Water(cm)'],
                       valid_mac_pw['effective_mac'])
slope, intercept, _, _, se = stats.linregress(
    valid_mac_pw['Precipitable_Water(cm)'], valid_mac_pw['effective_mac'])
x_fit = np.linspace(valid_mac_pw['Precipitable_Water(cm)'].min(),
                     valid_mac_pw['Precipitable_Water(cm)'].max(), 100)
ax.plot(x_fit, slope * x_fit + intercept, 'k-', linewidth=2.5,
        label=f'MAC = {slope:.1f}·PW + {intercept:.1f}')
ax.fill_between(x_fit,
                (slope - 1.96*se) * x_fit + intercept,
                (slope + 1.96*se) * x_fit + intercept,
                alpha=0.15, color='gray')

ax.axhline(y=MAC_VALUE, color='red', linestyle='--', alpha=0.7,
           label=f'Default MAC = {MAC_VALUE}')
ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.2e}\nn = {len(valid_mac_pw)}\n'
        f'R² = {r**2:.3f}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax.set_xlabel('Precipitable Water (cm)', fontsize=12)
ax.set_ylabel('Effective MAC (m²/g)', fontsize=12)
ax.set_title('Effective MAC vs Column Moisture\n(the money plot)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)

# --- Panel 2: MAC by season (box + swarm) ---
ax = axes[1]
plot_data_mac = [valid_mac_pw[valid_mac_pw['season'] == s]['effective_mac'] for s in SEASON_ORDER]
bp = ax.boxplot(plot_data_mac, labels=SEASON_ORDER, patch_artist=True,
               showfliers=False, widths=0.6)
for i, (patch, season) in enumerate(zip(bp['boxes'], SEASON_ORDER)):
    patch.set_facecolor(SEASON_COLORS[season])
    patch.set_alpha(0.7)
    sdata = valid_mac_pw[valid_mac_pw['season'] == season]['effective_mac']
    x_jitter = np.random.normal(i + 1, 0.08, len(sdata))
    ax.scatter(x_jitter, sdata, color=SEASON_COLORS[season],
              alpha=0.4, s=15, edgecolors='k', linewidth=0.2)
    ax.text(i + 1, ax.get_ylim()[0] if i == 0 else sdata.max() * 1.05,
            f'n={len(sdata)}\nmed={sdata.median():.1f}',
            ha='center', fontsize=9, va='bottom')

ax.axhline(y=MAC_VALUE, color='red', linestyle='--', alpha=0.7, label=f'Default MAC={MAC_VALUE}')
ax.set_ylabel('Effective MAC (m²/g)', fontsize=12)
ax.set_title('MAC by Ethiopian Season', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# --- Panel 3: PW by season (for context) ---
ax = axes[2]
plot_data_pw = [valid_mac_pw[valid_mac_pw['season'] == s]['Precipitable_Water(cm)'] for s in SEASON_ORDER]
bp = ax.boxplot(plot_data_pw, labels=SEASON_ORDER, patch_artist=True,
               showfliers=False, widths=0.6)
for i, (patch, season) in enumerate(zip(bp['boxes'], SEASON_ORDER)):
    patch.set_facecolor(SEASON_COLORS[season])
    patch.set_alpha(0.7)
    sdata = valid_mac_pw[valid_mac_pw['season'] == season]['Precipitable_Water(cm)']
    x_jitter = np.random.normal(i + 1, 0.08, len(sdata))
    ax.scatter(x_jitter, sdata, color=SEASON_COLORS[season],
              alpha=0.4, s=15, edgecolors='k', linewidth=0.2)
    ax.text(i + 1, sdata.max() * 1.02, f'n={len(sdata)}\nmed={sdata.median():.2f}',
            ha='center', fontsize=9, va='bottom')

ax.set_ylabel('Precipitable Water (cm)', fontsize=12)
ax.set_title('Columnar Moisture by Season\n(drives the MAC variation)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Precipitable Water Controls Effective MAC at Addis Ababa',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Print seasonal MAC values
print('\nSeasonal effective MAC (m²/g):')
for season in SEASON_ORDER:
    s = valid_mac_pw[valid_mac_pw['season'] == season]['effective_mac']
    pw = valid_mac_pw[valid_mac_pw['season'] == season]['Precipitable_Water(cm)']
    print(f'  {season}: MAC mean={s.mean():.1f}, median={s.median():.1f}, '
          f'std={s.std():.1f}, n={len(s)}  |  PW mean={pw.mean():.2f} cm')

print(f'\nOverall regression: MAC = {slope:.2f} × PW + {intercept:.2f}')
print(f'  At PW=1.0 cm (dry): MAC = {slope * 1.0 + intercept:.1f}')
print(f'  At PW=3.0 cm (wet): MAC = {slope * 3.0 + intercept:.1f}')
print(f'  At PW=2.0 cm (mean): MAC = {slope * 2.0 + intercept:.1f}')

In [ ]:
# Residual analysis: how good is the PW-MAC linear model?
valid_mac_pw = filt.dropna(subset=['effective_mac', 'Precipitable_Water(cm)']).copy()
mac_q01 = valid_mac_pw['effective_mac'].quantile(0.01)
mac_q99 = valid_mac_pw['effective_mac'].quantile(0.99)
valid_mac_pw = valid_mac_pw[
    (valid_mac_pw['effective_mac'] >= mac_q01) &
    (valid_mac_pw['effective_mac'] <= mac_q99)]

slope, intercept, _, _, _ = stats.linregress(
    valid_mac_pw['Precipitable_Water(cm)'], valid_mac_pw['effective_mac'])
valid_mac_pw['mac_predicted'] = slope * valid_mac_pw['Precipitable_Water(cm)'] + intercept
valid_mac_pw['mac_residual'] = valid_mac_pw['effective_mac'] - valid_mac_pw['mac_predicted']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Residuals vs PW (check linearity)
ax = axes[0]
for season in SEASON_ORDER:
    s = valid_mac_pw[valid_mac_pw['season'] == season]
    ax.scatter(s['Precipitable_Water(cm)'], s['mac_residual'],
              color=SEASON_COLORS[season], alpha=0.5, s=40, label=season)
ax.axhline(y=0, color='k', linestyle='--')
ax.set_xlabel('Precipitable Water (cm)', fontsize=11)
ax.set_ylabel('MAC residual (observed − predicted)', fontsize=11)
ax.set_title('Residuals vs PW\n(check for non-linearity)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: QQ plot of residuals
ax = axes[1]
resid_sorted = np.sort(valid_mac_pw['mac_residual'].dropna())
theoretical = stats.norm.ppf(np.linspace(0.01, 0.99, len(resid_sorted)))
ax.scatter(theoretical, resid_sorted, alpha=0.4, s=15, color='#2C3E50')
lims = [min(theoretical.min(), resid_sorted.min()),
        max(theoretical.max(), resid_sorted.max())]
ax.plot(lims, lims, 'r--', linewidth=2)
ax.set_xlabel('Theoretical quantiles', fontsize=11)
ax.set_ylabel('Sample quantiles', fontsize=11)
ax.set_title('QQ Plot of MAC Residuals', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

# Panel 3: Improvement from PW-adjusted MAC
ax = axes[2]
# Error using default MAC=10 vs PW-adjusted
default_ec = valid_mac_pw['hips_fabs']  # already /10
ftir_ec = valid_mac_pw['ftir_ec']
# PW-adjusted EC: HIPS_Fabs / predicted_MAC = (hips_fabs * MAC_VALUE) / predicted_MAC
pw_adjusted_ec = (valid_mac_pw['hips_fabs'] * MAC_VALUE) / valid_mac_pw['mac_predicted']

err_default = default_ec - ftir_ec
err_adjusted = pw_adjusted_ec - ftir_ec

ax.hist(err_default.dropna(), bins=40, alpha=0.5, color='#E74C3C',
        label=f'Default MAC={MAC_VALUE}\nRMSE={np.sqrt((err_default**2).mean()):.3f}', density=True)
ax.hist(err_adjusted.dropna(), bins=40, alpha=0.5, color='#27AE60',
        label=f'PW-adjusted MAC\nRMSE={np.sqrt((err_adjusted**2).mean()):.3f}', density=True)
ax.axvline(x=0, color='k', linestyle='--')
ax.set_xlabel('HIPS EC − FTIR EC (µg/m³)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('EC Estimation Error:\nDefault vs PW-Adjusted MAC', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('MAC–PW Model Diagnostics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

rmse_default = np.sqrt((err_default**2).mean())
rmse_adjusted = np.sqrt((err_adjusted**2).mean())
print(f'RMSE with default MAC={MAC_VALUE}: {rmse_default:.4f} µg/m³')
print(f'RMSE with PW-adjusted MAC: {rmse_adjusted:.4f} µg/m³')
print(f'Improvement: {(1 - rmse_adjusted/rmse_default)*100:.1f}%')

---

## 2. FTIR vs HIPS Correlation Gap with AOD

From the deep dive: FTIR EC correlates with AOD at r=0.661 while HIPS correlates at r=0.521 — a gap of 0.14. This is direct evidence that the **thermal method (FTIR EC) tracks the true aerosol column loading more faithfully than the optical method (HIPS)** at this site.

We formally test whether these r-values are statistically different using the Steiger (1980) test for comparing correlated correlation coefficients.

In [ ]:
def steiger_test(r_xy, r_xz, r_yz, n):
    """
    Steiger (1980) test for comparing two dependent correlations
    that share one variable.
    Tests H0: r(X,Y) = r(X,Z) where Y and Z are correlated.
    
    r_xy: correlation between X and Y (AOD vs FTIR EC)
    r_xz: correlation between X and Z (AOD vs HIPS EC)
    r_yz: correlation between Y and Z (FTIR EC vs HIPS EC)
    n: sample size
    
    Returns: z-statistic, p-value (two-tailed)
    """
    # Fisher z-transform
    z_xy = np.arctanh(r_xy)
    z_xz = np.arctanh(r_xz)
    
    # Determinant of the 3x3 correlation matrix
    r_bar = (r_xy + r_xz) / 2
    det = (1 - r_xy**2 - r_xz**2 - r_yz**2 + 2*r_xy*r_xz*r_yz)
    
    # Steiger formula
    num = (z_xy - z_xz) * np.sqrt((n - 3) / 2)
    denom = np.sqrt(1 - r_yz**2 + ((r_bar**2) * (1 - 2*r_yz + r_yz**2) / (1 - r_bar**2)))
    
    # Simplified Williams (1959) modification used in practice
    # More robust: use Zou (2007) confidence interval approach
    z_stat = (z_xy - z_xz) * np.sqrt((n - 3) * (1 + r_yz) / 
              (2 * (1 - r_xy**2 - r_xz**2 - r_yz**2 + 2*r_xy*r_xz*r_yz)))
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    return z_stat, p_value

# Get the three correlations from our merged filter+AERONET data
valid_trio = filt.dropna(subset=['ftir_ec', 'hips_fabs', 'AOD_500nm']).copy()
valid_trio = valid_trio[valid_trio['ftir_ec'] > 0]
n = len(valid_trio)

r_aod_ftir, p_aod_ftir = stats.pearsonr(valid_trio['AOD_500nm'], valid_trio['ftir_ec'])
r_aod_hips, p_aod_hips = stats.pearsonr(valid_trio['AOD_500nm'], valid_trio['hips_fabs'])
r_ftir_hips, p_ftir_hips = stats.pearsonr(valid_trio['ftir_ec'], valid_trio['hips_fabs'])

z_stat, p_steiger = steiger_test(r_aod_ftir, r_aod_hips, r_ftir_hips, n)

print('Correlation with AOD 500nm:')
print(f'  FTIR EC vs AOD:  r = {r_aod_ftir:.3f}  (p = {p_aod_ftir:.2e})')
print(f'  HIPS EC vs AOD:  r = {r_aod_hips:.3f}  (p = {p_aod_hips:.2e})')
print(f'  Gap: Δr = {r_aod_ftir - r_aod_hips:.3f}')
print(f'\nFTIR EC vs HIPS EC: r = {r_ftir_hips:.3f}')
print(f'\nSteiger test for dependent correlations:')
print(f'  z = {z_stat:.3f}, p = {p_steiger:.4f}')
print(f'  n = {n} samples')
if p_steiger < 0.05:
    print(f'  → FTIR EC correlates SIGNIFICANTLY better with AOD than HIPS EC does')
else:
    print(f'  → Difference is not statistically significant at p<0.05')

In [ ]:
# Visual comparison: FTIR EC vs AOD  |  HIPS EC vs AOD
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, (col, label, r_val, p_val) in enumerate([
    ('ftir_ec', 'FTIR EC (µg/m³)', r_aod_ftir, p_aod_ftir),
    ('hips_fabs', 'HIPS EC (µg/m³, MAC=10)', r_aod_hips, p_aod_hips),
]):
    ax = axes[i]
    for season in SEASON_ORDER:
        s = valid_trio[valid_trio['season'] == season]
        ax.scatter(s[col], s['AOD_500nm'], color=SEASON_COLORS[season],
                  alpha=0.6, s=50, label=season, edgecolors='k', linewidth=0.3)
    
    # Regression line
    sl, ic, _, _, _ = stats.linregress(valid_trio[col], valid_trio['AOD_500nm'])
    x_fit = np.linspace(valid_trio[col].min(), valid_trio[col].max(), 50)
    ax.plot(x_fit, sl * x_fit + ic, 'k-', linewidth=2.5)
    
    ax.text(0.05, 0.95, f'r = {r_val:.3f}\np = {p_val:.2e}\nn = {n}',
            transform=ax.transAxes, fontsize=11, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.set_xlabel(label, fontsize=12)
    ax.set_ylabel('AOD 500nm', fontsize=12)
    ax.set_title(f'{label.split("(")[0].strip()} vs AOD', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Panel 3: Direct HIPS vs FTIR scatter
ax = axes[2]
for season in SEASON_ORDER:
    s = valid_trio[valid_trio['season'] == season]
    ax.scatter(s['ftir_ec'], s['hips_fabs'], color=SEASON_COLORS[season],
              alpha=0.6, s=50, label=season, edgecolors='k', linewidth=0.3)

lims = [0, max(valid_trio['ftir_ec'].max(), valid_trio['hips_fabs'].max()) * 1.1]
ax.plot(lims, lims, 'k--', alpha=0.5, label='1:1 line')
sl, ic, _, _, _ = stats.linregress(valid_trio['ftir_ec'], valid_trio['hips_fabs'])
x_fit = np.linspace(0, lims[1], 50)
ax.plot(x_fit, sl * x_fit + ic, 'r-', linewidth=2.5,
        label=f'y = {sl:.2f}x + {ic:.3f}')
ax.text(0.55, 0.15, f'r = {r_ftir_hips:.3f}\nslope = {sl:.2f}\nintercept = {ic:.3f}',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax.set_xlabel('FTIR EC (µg/m³)', fontsize=12)
ax.set_ylabel('HIPS EC (µg/m³, MAC=10)', fontsize=12)
ax.set_title('HIPS vs FTIR Direct Comparison\n(slope & intercept = the two problems)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.suptitle(f'FTIR EC Tracks Columnar AOD Better Than HIPS EC (Δr = {r_aod_ftir - r_aod_hips:.3f}, '
             f'Steiger p = {p_steiger:.3f})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 3. Morning-Weighting Bias on 24h Filters

The diurnal decoupling shows BC is ~4x higher in the morning (6–10am) vs afternoon. Since the filter sampler runs continuously for 24 hours, the **mass loading is proportional to concentration × time**. If 50%+ of the mass is deposited in those 4 morning hours, it means the optical properties measured by HIPS are dominated by morning aerosol — which has different absorption characteristics due to the shallow boundary layer (higher concentration, potentially different mixing state and coating).

This connects boundary layer dynamics directly to why HIPS sees a different signal than FTIR.

In [ ]:
# Calculate what fraction of 24h BC mass comes from each time window
# Use minute-level BC data for each day that has filter data

bc_local = bc_raw[['IR BCc']].copy()
bc_local = bc_local.dropna()

# For each day, compute 24h total and 6-10am fraction
# Filter sampling is typically 9am-to-9am, so let's also check that window
daily_fractions = []

# Get all unique dates in BC data
bc_local['date'] = bc_local.index.date
bc_local['hour'] = bc_local.index.hour

for date, day_data in bc_local.groupby('date'):
    if len(day_data) < 60 * 18:  # need ≥18 hours of data
        continue
    
    total_mass = day_data['IR BCc'].sum()  # proportional to mass (concentration × 1 min)
    if total_mass <= 0:
        continue
    
    # Time windows
    early_morning = day_data[day_data['hour'].between(6, 9)]['IR BCc'].sum()  # 6-10am
    late_morning = day_data[day_data['hour'].between(10, 13)]['IR BCc'].sum()  # 10am-2pm
    afternoon = day_data[day_data['hour'].between(14, 17)]['IR BCc'].sum()  # 2-6pm
    evening = day_data[day_data['hour'].between(18, 21)]['IR BCc'].sum()  # 6-10pm
    night = day_data[(day_data['hour'] >= 22) | (day_data['hour'] <= 5)]['IR BCc'].sum()  # 10pm-6am
    
    month = pd.Timestamp(date).month
    daily_fractions.append({
        'date': date,
        'month': month,
        'season': assign_season(month),
        'total_mass_proxy': total_mass,
        'f_early_morning': early_morning / total_mass,  # 6-10am
        'f_late_morning': late_morning / total_mass,
        'f_afternoon': afternoon / total_mass,
        'f_evening': evening / total_mass,
        'f_night': night / total_mass,
        'n_minutes': len(day_data),
        'mean_bc': day_data['IR BCc'].mean(),
        'morning_bc': day_data[day_data['hour'].between(6, 9)]['IR BCc'].mean(),
        'afternoon_bc': day_data[day_data['hour'].between(12, 16)]['IR BCc'].mean(),
    })

frac_df = pd.DataFrame(daily_fractions)
frac_df['date'] = pd.to_datetime(frac_df['date'])
frac_df['morning_ratio'] = frac_df['morning_bc'] / frac_df['afternoon_bc']

print(f'Days with ≥18h of BC data: {len(frac_df)}')
print(f'\nMean fraction of 24h BC mass from 6-10am (4 hours = 16.7% of day):')
print(f'  Overall: {frac_df["f_early_morning"].mean()*100:.1f}%')
for season in SEASON_ORDER:
    s = frac_df[frac_df['season'] == season]
    print(f'  {season}: {s["f_early_morning"].mean()*100:.1f}% (n={len(s)})')

In [ ]:
# Morning-weighting visualization
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Stacked bar of mass fraction by time window
ax = axes[0]
windows = ['f_early_morning', 'f_late_morning', 'f_afternoon', 'f_evening', 'f_night']
window_labels = ['6-10am', '10am-2pm', '2-6pm', '6-10pm', '10pm-6am']
window_colors = ['#E74C3C', '#F39C12', '#F1C40F', '#3498DB', '#2C3E50']
window_hours = [4, 4, 4, 4, 8]  # hours in each window

bottoms = np.zeros(len(SEASON_ORDER))
for win, label, color, hrs in zip(windows, window_labels, window_colors, window_hours):
    values = [frac_df[frac_df['season'] == s][win].mean() * 100 for s in SEASON_ORDER]
    bars = ax.bar(SEASON_ORDER, values, bottom=bottoms, color=color, alpha=0.8,
                  label=f'{label} ({hrs}h = {hrs/24*100:.0f}%)', edgecolor='k', linewidth=0.3)
    # Annotate if fraction > 10%
    for j, (v, b) in enumerate(zip(values, bottoms)):
        if v > 8:
            ax.text(j, b + v/2, f'{v:.0f}%', ha='center', va='center', fontsize=9, fontweight='bold')
    bottoms += values

ax.axhline(y=16.7, color='red', linestyle='--', alpha=0.5,
           label='Expected 6-10am if uniform (16.7%)')
ax.set_ylabel('% of 24h BC mass', fontsize=12)
ax.set_title('Where Does the Filter Mass Come From?\n(by time of day)', fontsize=13, fontweight='bold')
ax.legend(fontsize=7, loc='upper right', bbox_to_anchor=(1.0, 0.98))
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 105)

# Panel 2: Early morning fraction distribution by season
ax = axes[1]
for season in SEASON_ORDER:
    s = frac_df[frac_df['season'] == season]['f_early_morning'] * 100
    ax.hist(s, bins=20, alpha=0.5, color=SEASON_COLORS[season],
            label=f'{season} (mean={s.mean():.0f}%)', density=True)

ax.axvline(x=16.7, color='red', linestyle='--', linewidth=2,
           label='Expected if uniform (16.7%)')
ax.set_xlabel('% of 24h BC mass from 6-10am', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('6-10am Mass Fraction Distribution\n(how concentrated is the morning peak?)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: Morning/afternoon BC ratio vs HIPS/FTIR ratio
# Merge morning dominance with filter data
ax = axes[2]
frac_for_merge = frac_df[['date', 'f_early_morning', 'morning_ratio']].copy()
frac_for_merge['date'] = frac_for_merge['date'].dt.normalize()

filt_morning = filt.copy()
filt_morning['date_norm'] = pd.to_datetime(filt_morning['date']).dt.normalize()
filt_morning = filt_morning.merge(frac_for_merge, left_on='date_norm',
                                   right_on='date', how='left', suffixes=('', '_frac'))

valid_morn = filt_morning.dropna(subset=['f_early_morning', 'hips_ftir_ratio'])
if len(valid_morn) > 5:
    for season in SEASON_ORDER:
        s = valid_morn[valid_morn['season'] == season]
        ax.scatter(s['f_early_morning'] * 100, s['hips_ftir_ratio'],
                  color=SEASON_COLORS[season], alpha=0.6, s=50,
                  label=season, edgecolors='k', linewidth=0.3)
    
    r, p = stats.pearsonr(valid_morn['f_early_morning'], valid_morn['hips_ftir_ratio'])
    ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.3f}\nn = {len(valid_morn)}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax.axhline(y=1, color='k', linestyle='--', alpha=0.5)
ax.axvline(x=16.7, color='red', linestyle='--', alpha=0.3)
ax.set_xlabel('% of 24h BC mass from 6-10am', fontsize=12)
ax.set_ylabel('HIPS EC / FTIR EC', fontsize=12)
ax.set_title('Morning Dominance vs HIPS-FTIR Discrepancy\n(does morning-weighting explain the bias?)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Morning-Weighting Bias: 24h Filters Are Dominated by 6-10am Aerosol',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nSummary: mass fractions by time window')
for win, label, hrs in zip(windows, window_labels, window_hours):
    expected = hrs / 24 * 100
    actual = frac_df[win].mean() * 100
    print(f'  {label} ({hrs}h): expected={expected:.1f}%, actual={actual:.1f}%, '
          f'overweight={actual/expected:.2f}x')

---

## 4. PW as Predictor of Slope vs Intercept

The HIPS-FTIR disagreement has two components:
- **Slope ≠ 1**: MAC is wrong → HIPS over/underestimates proportionally
- **Intercept ≠ 0**: a baseline absorption artifact (e.g. moisture on the filter)

Split by PW median: if **slope changes** between low-PW and high-PW days, moisture affects the MAC. If **intercept changes**, moisture creates a baseline artifact. If both change, it's a combination.

In [ ]:
# PW median split: HIPS Fabs vs FTIR EC in low vs high PW conditions
valid_split = filt.dropna(subset=['hips_fabs', 'ftir_ec', 'Precipitable_Water(cm)']).copy()
valid_split = valid_split[valid_split['ftir_ec'] > 0]

pw_median = valid_split['Precipitable_Water(cm)'].median()
valid_split['pw_group'] = np.where(
    valid_split['Precipitable_Water(cm)'] <= pw_median, 'Low PW', 'High PW')

print(f'PW median: {pw_median:.2f} cm')
print(f'Low PW: n={len(valid_split[valid_split["pw_group"]=="Low PW"])}, '
      f'PW range: {valid_split[valid_split["pw_group"]=="Low PW"]["Precipitable_Water(cm)"].min():.2f}'
      f' – {valid_split[valid_split["pw_group"]=="Low PW"]["Precipitable_Water(cm)"].max():.2f} cm')
print(f'High PW: n={len(valid_split[valid_split["pw_group"]=="High PW"])}, '
      f'PW range: {valid_split[valid_split["pw_group"]=="High PW"]["Precipitable_Water(cm)"].min():.2f}'
      f' – {valid_split[valid_split["pw_group"]=="High PW"]["Precipitable_Water(cm)"].max():.2f} cm')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

pw_colors = {'Low PW': '#E67E22', 'High PW': '#3498DB'}
pw_markers = {'Low PW': 'o', 'High PW': '^'}

# --- Panel 1: HIPS Fabs (raw absorption, not /MAC) vs FTIR EC by PW group ---
ax = axes[0]
valid_split['hips_fabs_raw'] = valid_split['hips_fabs'] * MAC_VALUE  # back to Fabs units

regressions = {}
for group in ['Low PW', 'High PW']:
    g = valid_split[valid_split['pw_group'] == group]
    ax.scatter(g['ftir_ec'], g['hips_fabs_raw'], marker=pw_markers[group],
              color=pw_colors[group], alpha=0.5, s=50, label=group,
              edgecolors='k', linewidth=0.3)
    
    sl, ic, r, p, se = stats.linregress(g['ftir_ec'], g['hips_fabs_raw'])
    regressions[group] = {'slope': sl, 'intercept': ic, 'r': r, 'p': p, 'se': se, 'n': len(g)}
    x_fit = np.linspace(0, g['ftir_ec'].max() * 1.1, 50)
    ax.plot(x_fit, sl * x_fit + ic, '-', color=pw_colors[group], linewidth=2.5)
    ax.text(0.55 if group == 'Low PW' else 0.05,
            0.35 if group == 'Low PW' else 0.15,
            f'{group}:\nslope={sl:.1f} (=MAC)\nintercept={ic:.3f}\nr={r:.3f}, n={len(g)}',
            transform=ax.transAxes, fontsize=9, va='top', color=pw_colors[group],
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

# Reference line: default MAC=10
lims = [0, valid_split['ftir_ec'].max() * 1.1]
ax.plot(lims, [MAC_VALUE * x for x in lims], 'k--', alpha=0.4, label=f'MAC={MAC_VALUE}')
ax.set_xlabel('FTIR EC (µg/m³)', fontsize=12)
ax.set_ylabel('HIPS Fabs (1/Mm)', fontsize=12)
ax.set_title('HIPS Fabs vs FTIR EC\n(slope = effective MAC)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, alpha=0.3)

# --- Panel 2: Same but HIPS EC (/ MAC=10) vs FTIR EC ---
ax = axes[1]
for group in ['Low PW', 'High PW']:
    g = valid_split[valid_split['pw_group'] == group]
    ax.scatter(g['ftir_ec'], g['hips_fabs'], marker=pw_markers[group],
              color=pw_colors[group], alpha=0.5, s=50, label=group,
              edgecolors='k', linewidth=0.3)
    
    sl, ic, r, p, _ = stats.linregress(g['ftir_ec'], g['hips_fabs'])
    x_fit = np.linspace(0, g['ftir_ec'].max() * 1.1, 50)
    ax.plot(x_fit, sl * x_fit + ic, '-', color=pw_colors[group], linewidth=2.5)
    ax.text(0.55 if group == 'Low PW' else 0.05,
            0.95 if group == 'Low PW' else 0.80,
            f'{group}:\nslope={sl:.2f}\nintercept={ic:.3f}\nr={r:.3f}',
            transform=ax.transAxes, fontsize=9, va='top', color=pw_colors[group],
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

lims = [0, max(valid_split['ftir_ec'].max(), valid_split['hips_fabs'].max()) * 1.1]
ax.plot(lims, lims, 'k--', alpha=0.4, label='1:1')
ax.set_xlabel('FTIR EC (µg/m³)', fontsize=12)
ax.set_ylabel('HIPS EC (µg/m³, MAC=10)', fontsize=12)
ax.set_title('HIPS EC vs FTIR EC by PW Group\n(1:1 = perfect MAC)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# --- Panel 3: Summary bar chart of slope and intercept ---
ax = axes[2]
x_pos = np.array([0, 1])
slopes = [regressions['Low PW']['slope'], regressions['High PW']['slope']]
intercepts = [regressions['Low PW']['intercept'], regressions['High PW']['intercept']]

ax2 = ax.twinx()
bars1 = ax.bar(x_pos - 0.15, slopes, 0.3, color=[pw_colors['Low PW'], pw_colors['High PW']],
               alpha=0.8, edgecolor='k', linewidth=0.5, label='Slope (= MAC)')
bars2 = ax2.bar(x_pos + 0.15, intercepts, 0.3, color=[pw_colors['Low PW'], pw_colors['High PW']],
                alpha=0.4, edgecolor='k', linewidth=0.5, hatch='//', label='Intercept')

ax.axhline(y=MAC_VALUE, color='red', linestyle='--', alpha=0.5, label=f'Default MAC={MAC_VALUE}')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Low PW\n(dry)', 'High PW\n(wet)'], fontsize=11)
ax.set_ylabel('Slope (effective MAC, m²/g)', fontsize=11, color='#2C3E50')
ax2.set_ylabel('Intercept (1/Mm)', fontsize=11, color='#7F8C8D')
ax.set_title('Slope (MAC) & Intercept by PW Group\n(which changes more?)', fontsize=13, fontweight='bold')

# Annotate
for i, (sl, ic) in enumerate(zip(slopes, intercepts)):
    ax.text(x_pos[i] - 0.15, sl + 0.2, f'{sl:.1f}', ha='center', fontsize=10, fontweight='bold')
    ax2.text(x_pos[i] + 0.15, ic + abs(ic)*0.1, f'{ic:.2f}', ha='center', fontsize=10, fontweight='bold')

h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'PW Median Split (threshold = {pw_median:.2f} cm): Does Moisture Affect Slope, Intercept, or Both?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Quantitative summary
print('\n--- PW Median Split Summary ---')
for group in ['Low PW', 'High PW']:
    r = regressions[group]
    print(f'{group}: slope (MAC) = {r["slope"]:.1f} m²/g, intercept = {r["intercept"]:.3f}, '
          f'r = {r["r"]:.3f}, n = {r["n"]}')

slope_change = regressions['High PW']['slope'] - regressions['Low PW']['slope']
intercept_change = regressions['High PW']['intercept'] - regressions['Low PW']['intercept']
print(f'\nSlope change (High − Low PW): {slope_change:+.1f} m²/g '
      f'({slope_change / regressions["Low PW"]["slope"] * 100:+.1f}%)')
print(f'Intercept change (High − Low PW): {intercept_change:+.3f}')

if abs(slope_change / regressions['Low PW']['slope']) > 0.1:
    print('→ Slope changes substantially: moisture affects MAC (absorption per unit mass)')
if abs(intercept_change) > 0.05:
    print('→ Intercept changes: moisture creates a baseline absorption artifact')

In [ ]:
# Tercile split for more granularity (low/mid/high PW)
pw_33 = valid_split['Precipitable_Water(cm)'].quantile(0.33)
pw_67 = valid_split['Precipitable_Water(cm)'].quantile(0.67)

valid_split['pw_tercile'] = pd.cut(
    valid_split['Precipitable_Water(cm)'],
    bins=[-np.inf, pw_33, pw_67, np.inf],
    labels=['Low PW', 'Mid PW', 'High PW'])

tercile_colors = {'Low PW': '#E67E22', 'Mid PW': '#27AE60', 'High PW': '#3498DB'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: HIPS Fabs vs FTIR EC by tercile
ax = axes[0]
for tercile in ['Low PW', 'Mid PW', 'High PW']:
    g = valid_split[valid_split['pw_tercile'] == tercile]
    ax.scatter(g['ftir_ec'], g['hips_fabs_raw'], color=tercile_colors[tercile],
              alpha=0.5, s=50, label=tercile, edgecolors='k', linewidth=0.3)
    if len(g) > 5:
        sl, ic, r, p, _ = stats.linregress(g['ftir_ec'], g['hips_fabs_raw'])
        x_fit = np.linspace(0, g['ftir_ec'].max() * 1.1, 50)
        ax.plot(x_fit, sl * x_fit + ic, '-', color=tercile_colors[tercile], linewidth=2.5)
        print(f'{tercile}: MAC={sl:.1f}, intercept={ic:.3f}, r={r:.3f}, '
              f'PW range=[{g["Precipitable_Water(cm)"].min():.2f}, '
              f'{g["Precipitable_Water(cm)"].max():.2f}], n={len(g)}')

lims = [0, valid_split['ftir_ec'].max() * 1.1]
ax.plot(lims, [MAC_VALUE * x for x in lims], 'k--', alpha=0.4, label=f'MAC={MAC_VALUE}')
ax.set_xlabel('FTIR EC (µg/m³)', fontsize=12)
ax.set_ylabel('HIPS Fabs (1/Mm)', fontsize=12)
ax.set_title('HIPS Fabs vs FTIR EC by PW Tercile\n(slope = MAC)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: Effective MAC distribution by tercile
ax = axes[1]
valid_split_mac = valid_split.dropna(subset=['effective_mac'])
for tercile in ['Low PW', 'Mid PW', 'High PW']:
    g = valid_split_mac[valid_split_mac['pw_tercile'] == tercile]['effective_mac']
    ax.hist(g, bins=20, alpha=0.5, color=tercile_colors[tercile],
            label=f'{tercile} (med={g.median():.1f})', density=True)

ax.axvline(x=MAC_VALUE, color='red', linestyle='--', linewidth=2, label=f'Default MAC={MAC_VALUE}')
ax.set_xlabel('Effective MAC (m²/g)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('MAC Distribution by PW Tercile', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('PW Tercile Analysis: Progressive MAC Change with Moisture',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 5. Synthesis: Toward a PW-Adjusted MAC

Bringing it all together:
1. PW explains ~23% of HIPS/FTIR variance (the strongest single predictor)
2. FTIR EC tracks the true aerosol column (AOD) more faithfully than HIPS
3. 6–10am accounts for a disproportionate fraction of 24h filter mass
4. PW affects the slope (MAC) and/or intercept of the HIPS-FTIR relationship

The physical story: moisture alters optical absorption properties on the HIPS filter but doesn't affect thermal EC (FTIR). This is why a single MAC doesn't work at Addis.

In [ ]:
# Final synthesis figure: the paper narrative
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Panel A: The money plot (MAC vs PW) ---
ax = axes[0, 0]
valid_synth = filt.dropna(subset=['effective_mac', 'Precipitable_Water(cm)']).copy()
q01 = valid_synth['effective_mac'].quantile(0.01)
q99 = valid_synth['effective_mac'].quantile(0.99)
valid_synth = valid_synth[(valid_synth['effective_mac'] >= q01) &
                          (valid_synth['effective_mac'] <= q99)]

for season in SEASON_ORDER:
    s = valid_synth[valid_synth['season'] == season]
    ax.scatter(s['Precipitable_Water(cm)'], s['effective_mac'],
              color=SEASON_COLORS[season], alpha=0.6, s=50,
              label=season, edgecolors='k', linewidth=0.3)

sl, ic, r, p, se = stats.linregress(
    valid_synth['Precipitable_Water(cm)'], valid_synth['effective_mac'])
x_fit = np.linspace(valid_synth['Precipitable_Water(cm)'].min(),
                     valid_synth['Precipitable_Water(cm)'].max(), 100)
ax.plot(x_fit, sl * x_fit + ic, 'k-', linewidth=2.5)
ax.axhline(y=MAC_VALUE, color='red', linestyle='--', alpha=0.7)
ax.text(0.05, 0.95, f'r = {r:.3f}, R² = {r**2:.3f}\np = {p:.2e}\nn = {len(valid_synth)}',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax.set_xlabel('Precipitable Water (cm)', fontsize=12)
ax.set_ylabel('Effective MAC (m²/g)', fontsize=12)
ax.set_title('(A) Column Moisture Drives MAC Variation', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Panel B: FTIR vs HIPS AOD correlation gap ---
ax = axes[0, 1]
valid_trio = filt.dropna(subset=['ftir_ec', 'hips_fabs', 'AOD_500nm'])
valid_trio = valid_trio[valid_trio['ftir_ec'] > 0]

r1, _ = stats.pearsonr(valid_trio['AOD_500nm'], valid_trio['ftir_ec'])
r2, _ = stats.pearsonr(valid_trio['AOD_500nm'], valid_trio['hips_fabs'])

for col, label, color, marker in [
    ('ftir_ec', f'FTIR EC (r={r1:.3f})', '#27AE60', 'o'),
    ('hips_fabs', f'HIPS EC (r={r2:.3f})', '#E74C3C', '^'),
]:
    ax.scatter(valid_trio[col], valid_trio['AOD_500nm'],
              color=color, marker=marker, alpha=0.4, s=40, label=label)
    sl, ic, _, _, _ = stats.linregress(valid_trio[col], valid_trio['AOD_500nm'])
    x_fit = np.linspace(valid_trio[col].min(), valid_trio[col].max(), 50)
    ax.plot(x_fit, sl * x_fit + ic, '-', color=color, linewidth=2.5)

ax.set_xlabel('EC (µg/m³)', fontsize=12)
ax.set_ylabel('AOD 500nm', fontsize=12)
ax.set_title(f'(B) FTIR Tracks AOD Better (Δr = {r1-r2:.3f})', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Panel C: Morning weighting ---
ax = axes[1, 0]
bottoms = np.zeros(len(SEASON_ORDER))
windows = ['f_early_morning', 'f_late_morning', 'f_afternoon', 'f_evening', 'f_night']
window_labels = ['6-10am', '10am-2pm', '2-6pm', '6-10pm', '10pm-6am']
window_colors = ['#E74C3C', '#F39C12', '#F1C40F', '#3498DB', '#2C3E50']

for win, label, color in zip(windows, window_labels, window_colors):
    values = [frac_df[frac_df['season'] == s][win].mean() * 100 for s in SEASON_ORDER]
    ax.bar(SEASON_ORDER, values, bottom=bottoms, color=color, alpha=0.8,
           label=label, edgecolor='k', linewidth=0.3)
    for j, (v, b) in enumerate(zip(values, bottoms)):
        if v > 10:
            ax.text(j, b + v/2, f'{v:.0f}%', ha='center', va='center',
                    fontsize=9, fontweight='bold')
    bottoms += values

ax.axhline(y=16.7, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('% of 24h BC mass', fontsize=12)
ax.set_title('(C) Morning Hours Dominate Filter Loading', fontsize=13, fontweight='bold')
ax.legend(fontsize=7, loc='upper right')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, axis='y')

# --- Panel D: PW split showing slope change ---
ax = axes[1, 1]
valid_pw_split = filt.dropna(subset=['hips_fabs', 'ftir_ec', 'Precipitable_Water(cm)']).copy()
valid_pw_split = valid_pw_split[valid_pw_split['ftir_ec'] > 0]
pw_med = valid_pw_split['Precipitable_Water(cm)'].median()
valid_pw_split['pw_group'] = np.where(
    valid_pw_split['Precipitable_Water(cm)'] <= pw_med, 'Low PW (dry)', 'High PW (wet)')
valid_pw_split['hips_fabs_raw'] = valid_pw_split['hips_fabs'] * MAC_VALUE

for group, color, marker in [('Low PW (dry)', '#E67E22', 'o'), ('High PW (wet)', '#3498DB', '^')]:
    g = valid_pw_split[valid_pw_split['pw_group'] == group]
    ax.scatter(g['ftir_ec'], g['hips_fabs_raw'], color=color, marker=marker,
              alpha=0.5, s=50, label=group, edgecolors='k', linewidth=0.3)
    if len(g) > 5:
        sl, ic, r, _, _ = stats.linregress(g['ftir_ec'], g['hips_fabs_raw'])
        x_fit = np.linspace(0, g['ftir_ec'].max() * 1.1, 50)
        ax.plot(x_fit, sl * x_fit + ic, '-', color=color, linewidth=2.5)
        ax.text(0.55 if 'Low' in group else 0.05,
                0.30 if 'Low' in group else 0.15,
                f'{group}:\nMAC={sl:.1f} m²/g\nr={r:.3f}',
                transform=ax.transAxes, fontsize=9, va='top', color=color,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

lims = [0, valid_pw_split['ftir_ec'].max() * 1.1]
ax.plot(lims, [MAC_VALUE * x for x in lims], 'k--', alpha=0.4, label=f'MAC={MAC_VALUE}')
ax.set_xlabel('FTIR EC (µg/m³)', fontsize=12)
ax.set_ylabel('HIPS Fabs (1/Mm)', fontsize=12)
ax.set_title(f'(D) Moisture Changes the MAC\n(PW split at {pw_med:.2f} cm)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('The Precipitable Water Story: Why a Single MAC Fails at Addis Ababa (2370m)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Final summary statistics
print('=' * 80)
print('SYNTHESIS: Precipitable Water & MAC at Addis Ababa')
print('=' * 80)

print(f'''
1. THE MONEY PLOT (MAC vs PW):
   Effective MAC = {sl:.1f} × PW + {ic:.1f}
   r = {r:.3f}, R² = {r**2:.3f}, p = {p:.2e}
   → PW explains {r**2*100:.0f}% of MAC variance
   → Dry season MAC ≈ {slope * 1.0 + intercept:.1f}, Kiremt MAC ≈ {slope * 3.0 + intercept:.1f}
''')

print(f'''2. CORRELATION GAP (FTIR vs HIPS with AOD):
   FTIR EC vs AOD: r = {r1:.3f}
   HIPS EC vs AOD: r = {r2:.3f}
   Δr = {r1-r2:.3f}
   → FTIR tracks columnar loading more faithfully
''')

morning_frac = frac_df['f_early_morning'].mean() * 100
print(f'''3. MORNING WEIGHTING:
   6-10am (4 hours = 16.7% of day) accounts for {morning_frac:.0f}% of filter mass
   Morning/afternoon BC ratio: {frac_df["morning_ratio"].median():.1f}x (median)
   → HIPS measures optical properties dominated by morning aerosol
''')

if regressions:
    print(f'''4. PW MEDIAN SPLIT (slope vs intercept):
   Low PW:  MAC = {regressions["Low PW"]["slope"]:.1f}, intercept = {regressions["Low PW"]["intercept"]:.3f}
   High PW: MAC = {regressions["High PW"]["slope"]:.1f}, intercept = {regressions["High PW"]["intercept"]:.3f}
   Δ slope = {slope_change:+.1f} m²/g ({slope_change / regressions["Low PW"]["slope"] * 100:+.0f}%)
   Δ intercept = {intercept_change:+.3f}
''')

print(f'''5. NARRATIVE FOR PAPER:
   "At Addis Ababa, HIPS-FTIR disagreement is driven by atmospheric moisture
   affecting optical absorption measurements. Column water vapor (from collocated
   AERONET) explains {r**2*100:.0f}% of the variance in effective MAC. Dust interference
   was ruled out (negative correlation). The site's extreme boundary layer dynamics
   at 2370m cause morning-dominated filter loading ({morning_frac:.0f}% from 6-10am),
   and seasonal moisture variation drives MAC from ~{regressions["Low PW"]["slope"]:.0f}
   (dry) to ~{regressions["High PW"]["slope"]:.0f} (Kiremt). A precipitable-water-adjusted
   MAC provides physically grounded site-specific calibration."
''')